In [1]:
# Imports
import random
import time

import librosa
import numpy as np
import jiwer
import pandas as pd
from whisper.normalizers.basic import BasicTextNormalizer
from tqdm.notebook import tqdm
from qwen_asr import Qwen3ASRModel

In [2]:
# Cargar modelo con backend vLLM
repo_id = "Qwen/Qwen3-ASR-1.7B"
print(f"Cargando modelo: {repo_id}")

model = Qwen3ASRModel.LLM(
    model=repo_id,
    gpu_memory_utilization=0.88,
    max_model_len=24576,
    swap_space=2,
    max_new_tokens=4096,
)

sr = 16000
CHUNK_SEC = 0.48

Cargando modelo: Qwen/Qwen3-ASR-1.7B
INFO 09-23 18:06:13 [utils.py:263] non-default args: {'max_model_len': 24576, 'swap_space': 2, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-ASR-1.7B'}
INFO 09-23 18:06:14 [model.py:530] Resolved architecture: Qwen3ASRForConditionalGeneration


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 09-23 18:06:15 [model.py:1545] Using max model len 24576
INFO 09-23 18:06:15 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-23 18:06:15 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 09-23 18:06:15 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


WARNING 09-23 18:06:16 [interface.py:470] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
WARNING 09-23 18:06:16 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=2150) INFO 09-23 18:06:26 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='Qwen/Qwen3-ASR-1.7B', speculative_config=None, tokenizer='Qwen/Qwen3-ASR-1.7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=24576, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto,

...

(EngineCore_DP0 pid=2150) INFO 09-23 18:06:41 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:25<00:25, 25.41s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:29<00:00, 12.85s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:29<00:00, 14.74s/it]
(EngineCore_DP0 pid=2150) 


(EngineCore_DP0 pid=2150) INFO 09-23 18:07:11 [default_loader.py:291] Loading weights took 29.49 seconds
(EngineCore_DP0 pid=2150) INFO 09-23 18:07:16 [gpu_model_runner.py:3905] Model loading took 3.87 GiB memory and 33.135821 seconds
(EngineCore_DP0 pid=2150) INFO 09-23 18:07:17 [gpu_model_runner.py:4715] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 21 audio items of the maximum feature size.
(EngineCore_DP0 pid=2150) INFO 09-23 18:07:31 [backends.py:644] Using cache directory: /home/carru/.cache/vllm/torch_compile_cache/330f7064d2/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2150) INFO 09-23 18:07:31 [backends.py:704] Dynamo bytecode transform time: 11.38 s
(EngineCore_DP0 pid=2150) INFO 09-23 18:07:39 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 3.347 s
(EngineCore_DP0 pid=2150) INFO 09-23 18:07:39 [monitor.py:34] torch.compile takes 14.72 s in total
(EngineCore_DP0 pid=2

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 12.84it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.36it/s]


(EngineCore_DP0 pid=2150) INFO 09-23 18:07:51 [gpu_model_runner.py:4856] Graph capturing finished in 7 secs, took 0.45 GiB
(EngineCore_DP0 pid=2150) INFO 09-23 18:07:51 [core.py:273] init engine (profile, create kv cache, warmup model) took 34.31 seconds
INFO 09-23 18:07:51 [llm.py:347] Supported tasks: ['generate', 'transcription']


In [3]:
N_AUDIOS = 20
SEMILLA = 42
DURACION_MAX_S = 90

IDIOMAS = {
    "es": ("../../Europarl-ST/es/en/test", "Europarl-ST.v2.es.en.test.lst", "Spanish"),
    "en": ("../../Europarl-ST/en/es/test", "Europarl-ST.v2.en.es.test.lst", "English"),
    "fr": ("../../Europarl-ST/fr/en/test", "Europarl-ST.fr.en.test.lst", "French"),
    "de": ("../../Europarl-ST/de/en/test", "Europarl-ST.de.en.test.lst", "German"),
}


def muestra(idioma):
    """Barajado con semilla fija; se toman los primeros que no exceden el tope.

    El tope existe porque el contexto de Qwen se agota en torno a los 120 s con
    bloques de 480 ms, y su coste crece de forma cuadratica con la duracion.
    """
    base, lst, _ = IDIOMAS[idioma]
    with open(base + "/" + lst, "r", encoding="utf-8") as f:
        todos = [l.strip() for l in f if l.strip()]

    random.Random(SEMILLA).shuffle(todos)

    sel = []
    for a in todos:
        d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
        if d <= DURACION_MAX_S:
            sel.append(a)
            if len(sel) == N_AUDIOS:
                break
    return sel


seleccion = {k: muestra(k) for k in IDIOMAS}
for k, v in seleccion.items():
    print(k, len(v), "audios")

/tmp/ipykernel_2084/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_2084/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_2084/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_2084/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duratio

es 20 audios
en 20 audios
fr 20 audios
de 20 audios


/tmp/ipykernel_2084/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)


In [4]:
def transcribir_offline(audio_array, lang):
    t0 = time.perf_counter()
    res = model.transcribe(audio=(audio_array, sr), language=lang)
    return res[0].text, time.perf_counter() - t0


def transcribir_streaming(audio_array, lang):
    estado = model.init_streaming_state(language=lang, chunk_size_sec=CHUNK_SEC)
    paso = int(CHUNK_SEC * sr)

    t0 = time.perf_counter()
    for i in range(0, len(audio_array), paso):
        model.streaming_transcribe(audio_array[i:i + paso], estado)
    model.finish_streaming_transcribe(estado)

    return estado.text, time.perf_counter() - t0

In [5]:
filas = []

for idioma in IDIOMAS:
    base, _, lang = IDIOMAS[idioma]

    for audio in tqdm(seleccion[idioma], desc=f"Qwen {idioma}"):

        with open(base + "/%s/transcription.tok" % audio, "r", encoding="utf-8") as f:
            ref = f.read()

        audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
        duracion = len(audio_array) / sr

        for modo, fn in [("offline", transcribir_offline), ("streaming", transcribir_streaming)]:
            try:
                hyp, total = fn(audio_array, lang)
                fallo = ""
            except Exception as e:
                hyp, total, fallo = "", np.nan, type(e).__name__
                print(f"  fallo {idioma}/{modo} en {audio} ({duracion:.0f} s): {str(e)[:80]}")

            filas.append(dict(idioma=idioma, audio=audio, modo=modo, fallo=fallo,
                              duracion_audio_s=duracion, tiempo_total_s=total,
                              rtf=total / duracion, hypothesis=hyp, reference=ref))

data = pd.DataFrame(filas)
data.to_csv("qwen_idiomas_raw.csv", index=False)
data.head()

Qwen es:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. 

Qwen en:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. 

  fallo en/streaming en en.20100309.26.2-474 (87 s): The decoder prompt (length 24578) is longer than the maximum model length of 245


...

Qwen fr:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. 

Qwen de:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2084/697181001.py:11: UserWarning: PySoundFile failed. 

,idioma,audio,modo,fallo,duracion_audio_s,tiempo_total_s,rtf,hypothesis,reference
0,es,en.20090324.3.2-058,offline,,89.025312,11.155981,0.125312,"Yo quisiera, respecto al informe sobre microcr...","Señora Presidenta , yo quisiera , respecto al ..."
1,es,en.20090324.3.2-058,streaming,,89.025312,48.676889,0.546776,Yo quisiera respecto al informe sobre microcré...,"Señora Presidenta , yo quisiera , respecto al ..."
2,es,en.20121025.28.4-369-000,offline,,65.015875,3.321977,0.051095,Yo también quiero felicitar al ponente y al re...,"Señor Presidente , yo también quiero felicitar..."
3,es,en.20121025.28.4-369-000,streaming,,65.015875,31.169718,0.479417,Yo también quiero felicitar al ponente y al re...,"Señor Presidente , yo también quiero felicitar..."
4,es,en.20101124.20.3-339,offline,,88.003687,4.290472,0.048753,También las gracias también a la señora Comisa...,"Señor Presidente , señora Comisaria , comparto..."


In [6]:
# Normalización de texto para evaluación
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(str(t)) for t in data["hypothesis"]]
data["reference_clean"] = [normalizer(str(t)) for t in data["reference"]]

data.to_csv("qwen_idiomas_limpio.csv", index=False)
data.head()

,idioma,audio,modo,fallo,duracion_audio_s,tiempo_total_s,rtf,hypothesis,reference,hypothesis_clean,reference_clean
0,es,en.20090324.3.2-058,offline,,89.025312,11.155981,0.125312,"Yo quisiera, respecto al informe sobre microcr...","Señora Presidenta , yo quisiera , respecto al ...",yo quisiera respecto al informe sobre microcré...,señora presidenta yo quisiera respecto al info...
1,es,en.20090324.3.2-058,streaming,,89.025312,48.676889,0.546776,Yo quisiera respecto al informe sobre microcré...,"Señora Presidenta , yo quisiera , respecto al ...",yo quisiera respecto al informe sobre microcré...,señora presidenta yo quisiera respecto al info...
2,es,en.20121025.28.4-369-000,offline,,65.015875,3.321977,0.051095,Yo también quiero felicitar al ponente y al re...,"Señor Presidente , yo también quiero felicitar...",yo también quiero felicitar al ponente y al re...,señor presidente yo también quiero felicitar a...
3,es,en.20121025.28.4-369-000,streaming,,65.015875,31.169718,0.479417,Yo también quiero felicitar al ponente y al re...,"Señor Presidente , yo también quiero felicitar...",yo también quiero felicitar al ponente y al re...,señor presidente yo también quiero felicitar a...
4,es,en.20101124.20.3-339,offline,,88.003687,4.290472,0.048753,También las gracias también a la señora Comisa...,"Señor Presidente , señora Comisaria , comparto...",también las gracias también a la señora comisa...,señor presidente señora comisaria comparto el ...


In [7]:
# WER offline frente a streaming, por idioma
resumen = []

for (idioma, modo), g in data.groupby(["idioma", "modo"]):
    ok = g[g["fallo"] == ""]
    resumen.append(dict(
        idioma=idioma,
        modo=modo,
        audios=len(ok),
        fallos=int((g["fallo"] != "").sum()),
        wer=jiwer.wer(list(ok["reference_clean"]), list(ok["hypothesis_clean"])) * 100,
        rtf=ok["rtf"].mean(),
    ))

resumen = pd.DataFrame(resumen).pivot(index="idioma", columns="modo",
                                      values=["wer", "rtf", "fallos"])
resumen.to_csv("qwen_idiomas_resumen.csv")
resumen

wer                  rtf            fallos          
modo      offline  streaming   offline streaming offline streaming
idioma                                                            
de      10.260994  12.191634  0.056918  0.492555     0.0       0.0
en       7.778738  15.573516  0.039047  0.454466     0.0       1.0
es       7.459893  10.802139  0.056710  0.520978     0.0       0.0
fr       6.214689  11.042630  0.052880  0.525635     0.0       0.0

In [8]:
# Degradación al pasar de offline a streaming
for idioma in IDIOMAS:
    g = data[data.idioma == idioma]
    off = g[(g.modo == "offline") & (g.fallo == "")]
    stm = g[(g.modo == "streaming") & (g.fallo == "")]
    if not len(off) or not len(stm):
        continue
    w_off = jiwer.wer(list(off.reference_clean), list(off.hypothesis_clean)) * 100
    w_stm = jiwer.wer(list(stm.reference_clean), list(stm.hypothesis_clean)) * 100
    print(f"{idioma}: offline {w_off:5.2f} %  ->  streaming {w_stm:5.2f} %  "
          f"(degradación {w_stm - w_off:+.2f} puntos, x{w_stm / w_off:.2f})")

es: offline  7.46 %  ->  streaming 10.80 %  (degradación +3.34 puntos, x1.45)
en: offline  7.78 %  ->  streaming 15.57 %  (degradación +7.79 puntos, x2.00)
fr: offline  6.21 %  ->  streaming 11.04 %  (degradación +4.83 puntos, x1.78)
de: offline 10.26 %  ->  streaming 12.19 %  (degradación +1.93 puntos, x1.19)
